# 02 — Exploration
Correlations, distributions, and the charts used in the findings/video. Outputs to `figures/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/clean.csv')
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False, 'axes.spines.right': False})
print(f"n = {len(df)}")

In [ ]:
# Correlations with GPA, ranked
numeric = df.select_dtypes('number')
corr = numeric.corr()['gpa'].drop('gpa').sort_values(key=abs, ascending=False)
corr.round(3)

In [ ]:
# Figure 1 — GPA by AI-verification habit
g = df.groupby('ai_verify')['gpa'].mean()
colors = ['#dc2626' if i == 1 else '#2563eb' for i in g.index]
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(g.index.astype(int).astype(str), g.values, color=colors)
ax.set_xlabel('How often do you verify AI answers? (1=Never, 5=Always)')
ax.set_ylabel('Average GPA'); ax.set_ylim(2.8, 3.8)
ax.set_title('Verification habit vs GPA')
plt.tight_layout(); plt.savefig('../figures/fig1_verify_vs_gpa.png', dpi=150)

In [ ]:
# Figure 2 — GPA by AI usage frequency (null result)
g = df.groupby('ai_freq')['gpa'].mean()
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(g.index.astype(int).astype(str), g.values, color='#f59e0b')
ax.set_xlabel('AI usage frequency (1=Never, 5=Very often)')
ax.set_ylabel('Average GPA'); ax.set_ylim(2.8, 3.8)
ax.set_title('Usage frequency vs GPA')
plt.tight_layout(); plt.savefig('../figures/fig2_frequency_null.png', dpi=150)

In [ ]:
# Figure 3 — GPA by weekly study hours
order = [2.5, 7.5, 15, 25, 35]; labels = ['0-5', '5-10', '10-20', '20-30', '30+']
g = df.groupby('study_hours_n')['gpa'].mean().reindex(order)
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(labels, g.values, color='#16a34a')
ax.set_xlabel('Hours studied per week'); ax.set_ylabel('Average GPA')
ax.set_ylim(2.8, 4.0); ax.set_title('Study hours vs GPA')
plt.tight_layout(); plt.savefig('../figures/fig3_hours_vs_gpa.png', dpi=150)

In [ ]:
# Figure 4 — correlation summary chart
labels_map = {'study_hours_n': 'Study hours/week', 'sleep_n': 'Sleep hours',
              'ai_verify': 'Verifying AI answers', 'consistency': 'Study consistency',
              'ai_dependence': 'AI dependence', 'ai_freq': 'AI usage frequency',
              'distraction': 'Distraction', 'independence': 'Working without help',
              'ai_confidence': 'AI confidence', 'difficulty': 'Course difficulty'}
c = df[list(labels_map) + ['gpa']].corr()['gpa'].drop('gpa').sort_values()
colors = ['#16a34a' if v > 0.05 else ('#dc2626' if v < -0.05 else '#9ca3af') for v in c.values]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([labels_map[k] for k in c.index], c.values, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Correlation with GPA'); ax.set_title('What correlates with GPA (early data)')
plt.tight_layout(); plt.savefig('../figures/fig4_correlations.png', dpi=150)

**Early takeaways (n≈55, patterns not proof):**
1. Study hours has the strongest positive correlation with GPA.
2. Never-verifying AI answers is associated with the lowest average GPA.
3. AI usage *frequency* shows ~zero correlation — how you use AI matters more than how much.

In [ ]:
# Helper: average-GPA bar chart with per-group sample sizes
def avg_chart(group_col, order, labels, xlabel, title, fname, color='#2563eb'):
    g = df.groupby(group_col)['gpa'].agg(['mean', 'count']).reindex(order)
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(labels, g['mean'], color=color)
    for b, v, n in zip(bars, g['mean'], g['count']):
        if pd.notna(v):
            ax.text(b.get_x()+b.get_width()/2, v+0.03, f'{v:.2f}', ha='center', fontweight='bold')
            ax.text(b.get_x()+b.get_width()/2, 2.85, f'n={int(n)}', ha='center', fontsize=10, color='#6b7280')
    ax.set_xlabel(xlabel); ax.set_ylabel('Average GPA'); ax.set_ylim(2.8, 4.0)
    ax.set_title(title)
    plt.tight_layout(); plt.savefig(f'../figures/{fname}', dpi=150)

In [ ]:
# Figure 5 — GPA by sleep (note: small n in extreme groups)
avg_chart('sleep_n', [4.5, 5.5, 6.5, 7.5, 8.5], ['<5', '5-6', '6-7', '7-8', '8+'],
          'Hours of sleep per night', 'Average GPA by sleep', 'fig5_sleep_vs_gpa.png', '#7c3aed')

In [ ]:
# Figure 6 — GPA by study consistency
avg_chart('consistency', [1, 2, 3, 4, 5], ['1', '2', '3', '4', '5'],
          'Study consistency (1=Very inconsistent, 5=Very consistent)',
          'Average GPA by study consistency', 'fig6_consistency_vs_gpa.png', '#0891b2')

In [ ]:
# Figure 7 — GPA by AI dependence
avg_chart('ai_dependence', [1, 2, 3, 4, 5], ['1', '2', '3', '4', '5'],
          'AI dependence for assignments (1=Not at all, 5=Highly)',
          'Average GPA by AI dependence', 'fig7_dependence_vs_gpa.png', '#f59e0b')

In [ ]:
# Figure 8 — average GPA by TYPE of AI use (multi-select, groups with n>=5)
use_cols = [c for c in df.columns if c.startswith('use_')]
rows = []
for c in use_cols:
    users = df[df[c] == 1]['gpa']
    if len(users) >= 5:
        rows.append((c.replace('use_', '').replace('_', ' ').title(), users.mean(), len(users)))
rows.sort(key=lambda r: r[1])
names, means, ns = zip(*rows)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, means, color='#2563eb')
for b, v, n in zip(bars, means, ns):
    ax.text(v + 0.005, b.get_y() + b.get_height()/2, f'{v:.2f} (n={n})', va='center', fontsize=11)
ax.set_xlim(3.2, 3.75); ax.set_xlabel('Average GPA of students who use AI this way')
ax.set_title('Average GPA by type of AI use')
plt.tight_layout(); plt.savefig('../figures/fig8_ai_use_types.png', dpi=150)

**Reading note:** treat any group with n < 10 as anecdotal — e.g. the '<5 hours sleep' group is only 2 students.